# M M 2 (max 200 m)

Twenty vessel arrivals, two service slots and a shared 200 m quay. The only Excel input is the **info** worksheet; all visits are generated below. The data worksheet is not read.

Customers (vessels) and the service station (quay) are composed of mix-ins, in the same way as objects in OpenTNSim.

In [ ]:
# package(s) for data handling
import pandas as pd
from IPython.display import display

# OpenQTSim mixins and utils
import openqtsim
from openqtsim.core import Identifiable, HasResource, HasLength, ExtraMetadata
from openqtsim.core import HasArrivalSchedule, HasHandlingTimes, HasMaxWait
from openqtsim.core import create_object, inspect_object
from openqtsim.visualizations import plot_gantt

print("This notebook is executed with OpenQTSim version {}".format(openqtsim.__version__))

#### 0. Define input

Use **exactly one** stopping limit: `nr_arrivals=20` and `max_duration_days=None`, or `nr_arrivals=None` and e.g. `max_duration_days=60`. A duration limit includes arrivals at the boundary; service and waiting then drain to completion or reneging, even beyond that limit.

`D` uses fixed times. `M` samples independent negative-exponential times. Stochastic first arrivals have mean `AT`; subsequent return intervals have mean `IAT`, separately for each vessel. Stochastic service has vessel-specific mean `loading + unloading`. Arrival and service seeds must be different; changing the service seed does not change the generated arrivals.

These are vessel-specific renewal streams and service mixtures, not necessarily a homogeneous textbook M/M/2 queue. Samples are generated in advance and replayed through the simulator's deterministic-table interface to preserve vessel identity.

In [ ]:
example_title = "M M 2 (max 200 m)"
nr_arrivals = 20
max_duration_days = None
arrival_distribution = "M"
service_distribution = "M"
arrival_seed = 42
service_seed = 43

#### 1. Create service station
Next we create a Quay-object (our service station) out of the available mix-ins:

In [ ]:
# make your preferred Quay class out of available mix-ins.
Quay = create_object(
    "Quay",
    (
        Identifiable,  # allows to give the object a name and a random ID
        HasResource,   # number of service slots (c in Kendall's notation)
        HasLength,     # quay length that is shared by the vessels at the quay
    ),
)

We can inspect the object to determine which inputs are required and optional:

In [ ]:
inspect_object(Quay, show_parameter_table=True)

We define the required input and create the quay object. A vessel is only admitted when both a service slot and enough remaining quay length are available:

In [ ]:
data_quay = {
    "name": "Quay",       # required by Identifiable
    "nr_resources": 2,    # optional for HasResource: two service slots
    "length": 200,        # required by HasLength: 200 m quay length
}

quay = Quay(**data_quay)

#### 2. Create vessels

All times in `info` are **days**, and lengths are metres. `AT` is the first arrival time (or its mean when stochastic); `IAT` is each vessel's return interval (or its mean). Service time is calculated from `loading + unloading`, not the redundant `ST` column. The generated simulation tables use **hours**.

In [ ]:
info = pd.read_excel('OpenQTSim - D D 1 example.xlsx', sheet_name='info')
info

We make a Vessel class out of the available mix-ins:

In [ ]:
# make your preferred Vessel class out of available mix-ins.
Vessel = create_object(
    "Vessel",
    (
        Identifiable,        # allows to give the object a name and a random ID
        HasLength,           # vessel length (m) that occupies the quay
        HasArrivalSchedule,  # first arrival AT and return interval IAT (days)
        HasHandlingTimes,    # unloading and loading time (days); service time is their sum
        HasMaxWait,          # maximum waiting time (days) before the vessel reneges
        ExtraMetadata,       # stores all other inputs (here the redundant ST column) as metadata
    ),
)

We can inspect the object to determine which inputs are required and optional:

In [ ]:
inspect_object(Vessel, show_parameter_table=True)

Each row of the info sheet is a data dict for one vessel object. The redundant `ST` column ends up in `vessel.metadata`:

In [ ]:
vessels = [Vessel(**data_vessel) for data_vessel in info.to_dict('records')]

pd.DataFrame([{
    "name": vessel.name, "length": vessel.length, "AT": vessel.AT, "IAT": vessel.IAT,
    "service_time": vessel.service_time, "maxwait": vessel.maxwait, "metadata": vessel.metadata,
} for vessel in vessels])

#### 3. Generate the arrival and service tables

Each row represents one visit. Visits are ordered by arrival time, with ties resolved by the original info-sheet row order. The simulator pairs both tables by row position. Subsequent visits remain scheduled even if an earlier visit reneges. `maxwait` is converted from days to hours, without randomisation.

In [ ]:
arrival_table, service_table = openqtsim.generate_vessel_tables(
    vessels,
    nr_arrivals=nr_arrivals,
    max_duration_days=max_duration_days,
    arrival_distribution=arrival_distribution,
    service_distribution=service_distribution,
    arrival_seed=arrival_seed,
    service_seed=service_seed,
)
print(f'Generated {len(arrival_table)} vessel arrivals; all table times are hours.')
display(arrival_table)
display(service_table)

#### 4. Run simulation

Admission requires both a free service slot and enough remaining length. Strict FIFO prevents bypassing a blocked vessel. A vessel that cannot start by `AT + maxwait` reneges and leaves the queue permanently. Service starting exactly at the deadline is allowed; zero patience allows immediate service only. The quay object sets the number of servers and the quay length of the queue.

In [ ]:
if arrival_table.empty:
    raise ValueError('No arrivals within the selected duration; increase the limit.')
A = openqtsim.ArrivalProcess('D', arrival_table)
S = openqtsim.ServiceProcess('D', service_table)
q = openqtsim.Queue(A, S, station=quay)
sim = openqtsim.Simulation(q)
sim.run(len(arrival_table))
summary = sim.get_stats()

In [ ]:
sim.plot_system_state()

#### 5. Inspect output

In [ ]:
df, df_sys = sim.return_log()
display(df)
display(df_sys.head(10))

#### 6. Gantt chart: arrival, waiting, service and reneging

One row per vessel visit, labelled by vessel name. Orange is waiting, blue is actual service, a green cross marks departure after service, and a **red cross** marks reneging. Reneged visits have no service bar. Hover labels show at most two decimals in hours, with days in parentheses. The table below includes status and actual departure time.

In [ ]:
fig = plot_gantt(df, title=f"{example_title}: {len(df)} vessel arrivals")
fig.show()
df.sort_values(["AT", "c_id"])[["c_id", "name", "length", "maxwait", "status", "AT", "TSB", "TSE",
                                "departure", "TCWQ", "ST", "TCSS", "s_id"]]